In [1]:
pip install psycopg2-binary


Note: you may need to restart the kernel to use updated packages.


**Connect to postgresql using your credentials**

In [1]:
import psycopg2

# Connexion à PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="demo",
    user="postgres",
    password="password"
)

# Create a cursor from the database connection.
# The cursor allows you to execute SQL queries and retrieve the results.
cursor = conn.cursor()


**Create tables and insert the data from the pdf**

In [2]:
cursor.execute("""
DROP TABLE IF EXISTS Doctor;
DROP TABLE IF EXISTS Hospital;

CREATE TABLE Hospital (
    Hospital_Id SERIAL PRIMARY KEY,
    Hospital_Name VARCHAR(100),
    Bed_Count INT
);

CREATE TABLE Doctor (
    Doctor_Id SERIAL PRIMARY KEY,
    Doctor_Name VARCHAR(100),
    Hospital_Id INT REFERENCES Hospital(Hospital_Id),
    Joining_Date DATE,
    Speciality VARCHAR(100),
    Salary INT,
    Experience INT
);
""")

# Commit all changes made in the table.
# Without commit(), INSERT, UPDATE, or DELETE operations are not permanently saved.
conn.commit()


In [3]:
# Insertion dans Hospital
#executemany()  runs the same query for multiple sets of values
# execute() runs the query once
cursor.executemany("""
INSERT INTO Hospital (Hospital_Id, Hospital_Name, Bed_Count)
VALUES (%s, %s, %s);
""", [
    (1, 'Mayo Clinic', 200),
    (2, 'Cleveland Clinic', 400),
    (3, 'Johns Hopkins', 1000),
    (4, 'Centre Médical', 1500)
])

# Insertion dans Doctor

cursor.executemany("""
INSERT INTO Doctor (Doctor_Id, Doctor_Name, Hospital_Id, Joining_Date, Speciality, Salary, Experience)
VALUES (%s, %s, %s, %s, %s, %s, %s);
""", [
    (101, 'David', 1, '2005-02-10', 'Pediatrie', 30000, None),
    (102, 'Michael', 1, '2018-07-23', 'Oncologue', 20000, None),
    (103, 'Susan', 2, '2016-05-19', 'Garnacologiste', 20000, None),
    (104, 'Robert', 2, '2017-12-28', 'Pediatrie', 28000, None),
    (105, 'Linda', 3, '2004-06-04', 'Garnacologiste', 42000, None),
    (106, 'William', 3, '2012-09-11', 'Dermatologue', 30000, None),
    (107, 'Richard', 4, '2014-11-19', 'Garnacologiste', 32000, None),
    (108, 'Karen', 4, '2011-10-17', 'Radiologue', 30000, None)
])
conn.commit()


**Exercise 1: Connect to your database server and print its version.**

In [4]:
#Sends a SQL command to the PostgreSQL server.
cursor.execute("SELECT version();")
# SELECT version(); retourne 1 row avec 1 colomn, fetchone() donne un Python tuple avec cette valeur.
print(cursor.fetchone())


('PostgreSQL 18.0 on x86_64-windows, compiled by msvc-19.44.35215, 64-bit',)


**Exercise 2: Retrieve information about the hospital and the doctor using the ID.**

In [4]:
hospital_id = 2
doctor_id = 103

cursor.execute("""
SELECT h.Hospital_Name, h.Bed_Count, d.Doctor_Name, d.Speciality
FROM Hospital h
JOIN Doctor d ON h.Hospital_Id = d.Hospital_Id
WHERE h.Hospital_Id = %s AND d.Doctor_Id = %s;
""", (hospital_id, doctor_id))

print(cursor.fetchall())


[('Cleveland Clinic', 400, 'Susan', 'Garnacologiste')]


%s placeholders are parameters in psycopg2.
When you pass (hospital_id, doctor_id) as the second argument, psycopg2 safely substitutes them into the query (2 and 103 here).

It prevents SQL injection. SQL injection is a code injection technique that might destroy your database. SQL injection is one of the most common web hacking techniques.

**Exercise 3: Obtain the list of doctors based on specialty and salary.**

In [5]:
speciality = 'Pediatrie'
min_salary = 25000

cursor.execute("""
SELECT Doctor_Name, Speciality, Salary
FROM Doctor
WHERE Speciality = %s AND Salary >= %s;
""", (speciality, min_salary))

print(cursor.fetchall())


[('David', 'Pediatrie', 30000), ('Robert', 'Pediatrie', 28000)]


**Explanation: filter doctors who are (Pediatrie)
And who earn at least 25,000.**

**Exercise 4: Obtain a list of doctors from a given hospital.**

In [6]:
hospital_id = 3

cursor.execute("""
SELECT Doctor_Name, Speciality, Salary
FROM Doctor
WHERE Hospital_Id = %s;
""", (hospital_id,))

print(cursor.fetchall())


[('Linda', 'Garnacologiste', 42000), ('William', 'Dermatologue', 30000)]


**Explanation: Get names of doctors from hospital id = 3**

**Exercise 5: Update the doctors' experience in years.**

In [7]:
cursor.execute("""
UPDATE Doctor
SET Experience = EXTRACT(YEAR FROM AGE(CURRENT_DATE, Joining_Date));
""")

#commit(), qui est utilisée pour valider les changements apportés à la base de données 
conn.commit()

# Vérifier
cursor.execute("SELECT Doctor_Name, Experience FROM Doctor;")
print(cursor.fetchall())


[('David', 21), ('Michael', 8), ('Susan', 10), ('Robert', 8), ('Linda', 22), ('William', 13), ('Richard', 11), ('Karen', 14)]


**AGE(CURRENT_DATE, Joining_Date)  gives the difference between today’s date and the doctor’s Joining_Date.

for exammple  if a doctor joined on 2015-06-01, AGE difference is 9 years 4 mons 1 day.
EXTRACT(YEAR FROM) gets only the year part from that result (so just 9).
SET Experience then updates and stores that value into the Experience column.**

**close() closes the database connection to release resources.**

In [8]:
#Ferme la connexion à la base de données.
cursor.close()
conn.close()


*Notes on rollback:

Cancels all uncommitted changes.

Useful in case of an error to go back to the previous state.

Example:

try:

    curseur.execute("UPDATE Doctor SET Salary = 'valeur_invalide' WHERE Doctor_Id = 101;")
    
    conn.commit()
    
except Exception:

    conn.rollback()  # Annule la modification échouée*
